# Processing gnomAD and ClinVar data

In [1]:
import os
import subprocess
import polars as pl
import pandas as pd
import re

# Set strict error handling
def run_command(cmd):
    result = subprocess.run(cmd, shell=True, check=True, text=True)
    return result

## 1. ClinVar

ClinVar variants released on 2024-03-01 and the data is accessed on March 13, 2025

### Clean up ClinVar annotations

For merging the ClinVar with the VarChAMP variants more easily

In [2]:
dest_path = "../1_inputs/raw_inputs/clinvar/variant_summary.txt.gz"
## Define the new processed and cleaned clinvar file
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"

In [4]:
# ## Read CSV with Polars (auto-decompresses .gz)
# clinvar = pl.read_csv(dest_path, separator='\t', infer_schema_length=1000000)

# ## Process data with Polars expressions
# clinvar = (
#     clinvar
#     .filter(pl.col("Assembly") == "GRCh38")
#     .with_columns(
#         pl.col("Name")
#         # Split the string on '.', '(' or ')'
#         .map_elements(lambda s: re.split(r"[.\(\)]", s), pl.List(pl.String))
#         .alias("Name_listed")
#     )
# )
# clinvar = clinvar.with_columns(
#     pl.col("Name_listed").list.get(0).alias("NM"),
#     pl.col("Name_listed").list.get(1).alias("version"), 
#     pl.col("Name_listed").list.get(2).alias("gene"),
#     pl.col("Name_listed").list.get(3).alias("c"),
#     pl.col("Name_listed").list.get(4).alias("clinvar_nt_change"),
#     pl.col("Name_listed").list.get(5).alias("p"),
#     pl.col("Name_listed").list.get(6).alias("clinvar_aa_change")
# ).with_columns(
#         SPDI=pl.col("ChromosomeAccession") + ":" + 
#             pl.col("PositionVCF").cast(str) + ":" +
#             pl.col("ReferenceAlleleVCF") + ":" +
#             pl.col("AlternateAlleleVCF"),
#         RefSeq_mRNA=pl.col("NM") + "." + pl.col("version"),
#         StarStatus=pl.when(pl.col("ReviewStatus").str.contains(r'single|conflicting'))
#             .then(1)
#             .when(pl.col("ReviewStatus").str.contains('multiple'))
#             .then(2)
#             .when(pl.col("ReviewStatus").str.contains('expert'))
#             .then(3)
#             .when(pl.col("ReviewStatus").str.contains('practice'))
#             .then(4)
#             .otherwise(None)
# )

In [4]:
## Create the mapping expression with precise logic for a cleaner ClinVar Annotation
clinvar_mapping = (
    # First check for Conflicting interpretations (highest priority)
    pl.when(
        pl.col("ClinicalSignificance").str.to_lowercase().str.contains("conflicting")
    )
    .then(pl.lit("3_Conflicting"))
    
    # Then check for Uncertain significance (VUS)
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("uncertain"))
    .then(pl.lit("4_VUS"))
    
    # # Special case for strings containing both "Uncertain significance" and "Pathogenic"
    # .when(
    #     pl.col("clinvar_clnsig").str.to_lowercase().str.contains("uncertain") &
    #     pl.col("clinvar_clnsig").str.to_lowercase().str.contains("pathogenic")
    # )
    # .then(pl.lit("4_VUS"))  # The dictionary says to classify this as VUS
    
    # Check for Pathogenic
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("pathogenic"))
    .then(pl.lit("1_Pathogenic"))
    
    # Check for Benign
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("benign"))
    .then(pl.lit("2_Benign"))
    
    # Everything else goes to Others
    .otherwise(pl.lit("5_Others"))
)

# Apply the mapping to create a new column
clinvar = clinvar.with_columns(clinvar_clnsig_clean=clinvar_mapping)

# Create a dataframe with unique mappings to check accuracy
unique_mappings = clinvar.select(
    pl.col("ClinicalSignificance"),
    pl.col("clinvar_clnsig_clean")
).unique()

# Sort by original value for easier verification
unique_mappings = unique_mappings.sort("ClinicalSignificance")

# Print each mapping to verify accuracy
print("Original Value -> Mapped Value")
print("-" * 50)
for row in unique_mappings.rows():
    print(f"{row[0]} -> {row[1]}")

Original Value -> Mapped Value
--------------------------------------------------
- -> 5_Others
Affects -> 5_Others
Affects; risk factor -> 5_Others
Benign -> 2_Benign
Benign/Likely benign -> 2_Benign
Benign/Likely benign; association -> 2_Benign
Benign/Likely benign; drug response -> 2_Benign
Benign/Likely benign; drug response; other -> 2_Benign
Benign/Likely benign; other -> 2_Benign
Benign/Likely benign; other; risk factor -> 2_Benign
Benign/Likely benign; risk factor -> 2_Benign
Benign; Affects -> 2_Benign
Benign; Affects; association; other -> 2_Benign
Benign; association -> 2_Benign
Benign; confers sensitivity -> 2_Benign
Benign; drug response -> 2_Benign
Benign; other -> 2_Benign
Benign; protective -> 2_Benign
Benign; risk factor -> 2_Benign
Conflicting classifications of pathogenicity -> 3_Conflicting
Conflicting classifications of pathogenicity; Affects -> 3_Conflicting
Conflicting classifications of pathogenicity; association -> 3_Conflicting
Conflicting classifications of p

In [5]:
# ## Make sure we only drop columns that actually exist
# columns_to_drop = [col for col in ['NM', 'version', 'gene', 'c', 'p', 'ReferenceAllele', 'AlternateAllele', 'Name_listed'] 
#                   if col in clinvar.columns]
# if columns_to_drop:
#     clinvar = clinvar.drop(columns_to_drop)

# # Now cast to categorical
# clinvar = clinvar.with_columns(
#     pl.col(["Assembly", "ClinicalSignificance", "ReviewStatus", "clinvar_clnsig_clean"]).cast(pl.Categorical)
# )

# # Write to Parquet with ZSTD compression
# clinvar.write_parquet(processed_clinvar_parq, 
#                      compression="zstd", 
#                      statistics=True)

## 2. Merge clinvar and gnomad to varchamp alleles

### 2.1 Loading varchamp alleles

In [70]:
## get unique info for genes in polars
gene_info = pl.scan_csv(f"../1_inputs/VarChamp_Consolidated_allele_collection.tsv", separator="\t", infer_schema_length=100000) #[gene_cols]
gene_info = gene_info.with_columns([
    # Fix: Use concat_str for string concatenation
    pl.concat_str([pl.lit("chr"), pl.col("chr")]).alias("chrom"),
    # Fix: Get the last character of the first part (before ">")
    pl.col("nt_change").str.split(">").list.get(0).str.slice(-1).alias("ref"),
    # Fix: Get the entire second part (after ">") - slice(0) gets from start to end
    pl.col("nt_change").str.split(">").list.get(-1).alias("alt")
]).with_columns([
    # Fix: Use concat_str for the varid string concatenation
    pl.concat_str([
        pl.col("chr"),
        pl.lit("-"),
        pl.col("chr_pos_38").cast(pl.String),
        pl.lit("-"),
        pl.col("ref"),
        pl.lit("-"),
        pl.col("alt")
    ]).alias("varid")
])

gene_info.collect()

orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,mutation_id_rc4,mutation_id_cegs2,mutation_id_edgotyping3,entry_plate,entry_well,collection,sequence,chrom,ref,alt,varid
i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2,"""GBA""","""160G>C""","""Val54Leu""","""ENSG00000177628""","""1""",155240033,"""NC_000001.11:155240033:C:G""","""ALE0000584""","""CCSBVarC000001""","""6""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""F12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""G""","""C""","""1-155240033-G-C"""
2,"""GBA""","""670C>T""","""Leu224Phe""","""ENSG00000177628""","""1""",155238225,"""NC_000001.11:155238225:G:A""","""ALE00000002""","""CCSBVarC000002""","""73""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""C12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""C""","""T""","""1-155238225-C-T"""
2,"""GBA""","""887G>A""","""Arg296Gln""","""ENSG00000177628""","""1""",155237453,"""NC_000001.11:155237453:C:T""","""ALE00000003""","""CCSBVarC000003""","""113""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""D12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""G""","""A""","""1-155237453-G-A"""
2,"""GBA""","""1448T>C""","""Leu483Pro""","""ENSG00000177628""","""1""",155235252,"""NC_000001.11:155235252:A:G""","""ALE00000004""","""CCSBVarC000004""","""231""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""E12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""T""","""C""","""1-155235252-T-C"""
25,"""SLC9A3R1""","""328C>G""","""Leu110Val""","""ENSG00000109062""","""17""",74749174,"""NC_000017.11:74749174:C:G""","""ALE00000005""","""CCSBVarC000005""","""252""","""200008""","""NULL""","""GDEhDisVCh_40045""","""E04""","""RC4""","""ATGAGCGCGGACGCAGCGGCCGGGGCGCCC…","""chr17""","""C""","""G""","""17-74749174-C-G"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
8999,"""TTR""","""355G>A""","""Asp119Asn""","""ENSG00000118271""","""18""",31598586,"""NC_000018.10:31598586:G:A""","""ALE00008468""","""CCSBVarC008481""","""NULL""","""NULL""","""390""","""NULL""","""NULL""","""Edgotyping3""","""ATGGCTTCTCATCGTCTGCTCCTCCTCTGC…","""chr18""","""G""","""A""","""18-31598586-G-A"""
7323,"""PMP22""","""152A>G""","""His51Arg""","""ENSG00000109099""","""17""",15259120,"""NC_000017.11:15259120:T:C""","""ALE00008469""","""CCSBVarC008466""","""NULL""","""NULL""","""394""","""NULL""","""NULL""","""Edgotyping3""","""ATGCTCCTCCTGTTGCTGAGTATCATCGTC…","""chr17""","""A""","""G""","""17-15259120-A-G"""
2651,"""CLN6""","""89G>C""","""Gly30Ala""","""ENSG00000128973""","""15""",68218645,"""NC_000015.10:68218645:C:G""","""ALE00008470""","""CCSBVarC008436""","""NULL""","""NULL""","""436""","""NULL""","""NULL""","""Edgotyping3""","""ATGGAGGCGACGCGGAGGCGGCAGCACCTG…","""chr15""","""G""","""C""","""15-68218645-G-C"""


### 2.2 Merge with ClinVar

In [71]:
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"
gene_info_clinvar_parq = "../3_outputs/varchamp_consolidated_alleles_clinvar_annot.parquet"
gene_info_clinvar_tsv = "../3_outputs/varchamp_consolidated_alleles_clinvar_annot.tsv"

clinvar = pl.scan_parquet(processed_clinvar_parq)
clinvar.collect().head()

gene_info_clinvar = gene_info.join(
                        clinvar,
                        left_on="spdi",
                        right_on="SPDI",
                        how="left"
                    )
gene_info_clinvar = gene_info_clinvar.collect()

# gene_info_clinvar.write_parquet(gene_info_clinvar_parq, compression="zstd")
# gene_info_clinvar.write_csv(gene_info_clinvar_tsv, separator="\t")

In [72]:
clinvar.collect().shape

(3390885, 47)

#### Report the number of alleles missing ClinVar annotations

In [73]:
gene_info_clinvar = pl.scan_parquet(gene_info_clinvar_parq)
gene_info_clinvar.filter((~pl.col("spdi").is_null())&(pl.col("#AlleleID").is_null())).collect()

orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,mutation_id_rc4,mutation_id_cegs2,mutation_id_edgotyping3,entry_plate,entry_well,collection,sequence,chrom,ref,alt,varid,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
2,"""GBA""","""160G>C""","""Val54Leu""","""ENSG00000177628""","""1""",155240033,"""NC_000001.11:155240033:C:G""","""ALE0000584""","""CCSBVarC000001""","""6""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""F12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""G""","""C""","""1-155240033-G-C""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""GBA""","""670C>T""","""Leu224Phe""","""ENSG00000177628""","""1""",155238225,"""NC_000001.11:155238225:G:A""","""ALE00000002""","""CCSBVarC000002""","""73""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""C12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""C""","""T""","""1-155238225-C-T""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
167,"""ACTA2""","""874A>G""","""Arg292Gly""","""ENSG00000107796""","""10""",88938177,"""NC_000010.11:88938177:T:C""","""ALE00000012""","""CCSBVarC000012""","""390""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""B12""","""RC4""","""ATGTGTGAAGAAGAGGACAGCACTGCCTTG…","""chr10""","""A""","""G""","""10-88938177-A-G""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
187,"""FOXA3""","""271G>A""","""Gly91Arg""","""ENSG00000170608""","""19""",45872276,"""NC_000019.10:45872276:G:A""","""ALE00000013""","""CCSBVarC000013""","""393""","""NULL""","""NULL""","""GDEhDisVCh_40008""","""G09""","""RC4""","""ATGCTGGGCTCAGTGAAGATGGAGGCCCAT…","""chr19""","""G""","""A""","""19-45872276-G-A""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
286,"""ABHD5""","""245A>G""","""His82Arg""","""ENSG00000011198""","""3""",43702326,"""NC_000003.12:43702326:A:G""","""ALE00000031""","""CCSBVarC000031""","""782""","""NULL""","""NULL""","""GDEhDisVCh_40077""","""H08""","""RC4""","""ATGGCGGCGGAGGAGGAGGAGGTGGACTCT…","""chr3""","""A""","""G""","""3-43702326-A-G""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

Check why missing, and it seems like some alleles just weren't collected by ClinVar yet

In [74]:
gene_info_clinvar.filter(pl.col("symbol")=="GBA").sort(by="clinvar_aa_change").collect()

orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,mutation_id_rc4,mutation_id_cegs2,mutation_id_edgotyping3,entry_plate,entry_well,collection,sequence,chrom,ref,alt,varid,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
2,"""GBA""","""160G>C""","""Val54Leu""","""ENSG00000177628""","""1""",155240033,"""NC_000001.11:155240033:C:G""","""ALE0000584""","""CCSBVarC000001""","""6""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""F12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""G""","""C""","""1-155240033-G-C""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""GBA""","""670C>T""","""Leu224Phe""","""ENSG00000177628""","""1""",155238225,"""NC_000001.11:155238225:G:A""","""ALE00000002""","""CCSBVarC000002""","""73""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""C12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""C""","""T""","""1-155238225-C-T""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""GBA""","""1483G>C""","""Ala495Pro""","""ENSG00000177628""","""1""",155235217,"""NC_000001.11:155235217:C:G""","""ALE00003889""","""CCSBVarC003889""","""NULL""","""213560""","""NULL""","""GDEhDisVCh_40054""","""G07""","""CEGS2""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""G""","""C""","""1-155235217-G-C""",38384,"""single nucleotide variant""","""NM_000157.4(GBA1):c.1483G>C (p…",2629,"""GBA1""","""HGNC:4177""","""Conflicting classifications of…",1,"""Aug 20, 2024""",368060,"""-""","""RCV000079341|RCV003129772""","""MedGen:CN169374|MedGen:C366190…","""not specified|not provided""","""germline""","""germline""","""GRCh38""","""NC_000001.11""","""1""",155235217,155235217,"""1q22""","""criteria provided, conflicting…",5,"""-""","""N""","""ClinGen:CA146980,UniProtKB:P04…",2,93450,155235217,"""C""","""G""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV002556287|SCV002756106|SCV0…","""-""","""-""","""1483G>C ""","""Ala495Pro""","""NM_000157.4""",1,"""3_Conflicting"""
2,"""GBA""","""476G>A""","""Arg159Gln""","""ENSG00000177628""","""1""",155238629,"""NC_000001.11:155238629:C:T""","""ALE00003870""","""CCSBVarC003870""","""NULL""","""213515""","""NULL""","""GDEhDisVCh_40054""","""E09""","""CEGS2""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""chr1""","""G""","""A""","""1-155238629-G-A""",19330,"""single nucleotide variant""","""NM_000157.4(GBA1):c.476G>A (p.…",2629,"""GBA1""","""HGNC:4177""","""Pathogenic/Likely pathogenic""",1,"""Jun 24, 2024""",79653797,"""-""","""RCV000004518|RCV000004519|RCV0…","""MONDO:MONDO:0009265,MedGen:C19…","""Gaucher disease type I|Gaucher…","""germline""","""germline""","""GRCh38""","""NC_000001.11""","""1""",155238629,155238629,"

### 2.3 Merge with gnomad

##### Sort out all the vcf files and all their position coordinates for looking up varchamp-related variants

In [18]:
## Generate all the position coordinates for varchamp-related variants for quicker look-up

## Define some constants
VCF_DIR = "../1_inputs/raw_inputs/gnomad"
CHROMOSOMES = [str(c) for c in list(range(1, 23))] + ['X', 'Y']

# VARIANT_LIST = f"../3_outputs/processed_data/gnomad_allele_frequencies/gnomad_variants_list.txt"
# gene_info_pd[["chrom","chr_pos_38","ref","alt"]].to_csv(VARIANT_LIST, sep="\t", header=False, index=False)

In [19]:
# # !conda install bcftools -c conda-forge -y ## install bcftools if you didn't

# OUTPUT_DIR = f"../3_outputs/processed_data/gnomad_allele_frequencies"
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Process chromosomes
# for chrom in CHROMOSOMES:
#     vcf_file = f"{VCF_DIR}/gnomad.exomes.v4.1.sites.chr{chrom}.vcf.bgz"
#     csi_file = f"{vcf_file}.csi"
#     output_file = f"{OUTPUT_DIR}/chr{chrom}_allele_frequencies.txt"
    
#     if os.path.exists(vcf_file):
#         print(f"Processing chromosome {chrom}...")
        
#         if not os.path.exists(csi_file):
#             print(f"Indexing {vcf_file}")
#             run_command(f"bcftools index --threads 4 {vcf_file}")
        
#         print(f"Extracting frequencies for {chrom}")
#         view_cmd = f"bcftools view -R {VARIANT_LIST} {vcf_file}"
#         query_cmd = "bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT\\t%AF\\n'"
        
#         with open(output_file, 'w') as f:
#             p1 = subprocess.Popen(view_cmd.split(), stdout=subprocess.PIPE)
#             p2 = subprocess.Popen(query_cmd.split(), stdin=p1.stdout, stdout=f)
#             p1.stdout.close()
#             p2.communicate()
            
#         print(f"Saved to {output_file}")
#     else:
#         print(f"Skipping chromosome {chrom} - file not found")

# print("Processing completed")

#### Merge the gnomad AF files from chromosomes

In [26]:
allele_freq_dfs = []
for chrom in CHROMOSOMES:
    try:
        allele_freq = pl.read_csv(f"../3_outputs/processed_data/gnomad_allele_frequencies/chr{chrom}_allele_frequencies.txt",
                                separator="\t", has_header=False, 
                                new_columns=["chr","chr_pos_38","ref","alt","allele_freq"])
        allele_freq = allele_freq.with_columns(
            pl.col("chr").str.replace(r"\'", "").str.replace(r"\'", "").str.replace("chr", "")
        ).drop_nulls()
        # allele_freq = allele_freq.filter(~pl.all_horizontal(pl.all().is_null()))
        # print(allele_freq)
        allele_freq_dfs.append(allele_freq)
    except:
        continue

allele_freq_all = pl.concat(allele_freq_dfs)
allele_freq_gnomad = allele_freq_all.with_columns(
    pl.concat_str([pl.col("chr"),
                   pl.lit('-'),
                   pl.col("chr_pos_38").cast(pl.String),
                   pl.lit('-'),
                   pl.col("ref"),
                   pl.lit('-'),
                   pl.col("alt")
                   ]).alias("varid"),
    pl.col("allele_freq").alias("gnomad_af")
).drop(pl.col("allele_freq"))#.to_pandas()

allele_freq_gnomad

chr,chr_pos_38,ref,alt,varid,gnomad_af
str,i64,str,str,str,f64
"""1""",2028259,"""C""","""A""","""1-2028259-C-A""",6.8500e-7
"""1""",2028259,"""C""","""T""","""1-2028259-C-T""",0.001409
"""1""",2028260,"""G""","""A""","""1-2028260-G-A""",0.0217404
"""1""",5907193,"""CG""","""C""","""1-5907193-CG-C""",0.0
"""1""",5907194,"""G""","""A""","""1-5907194-G-A""",0.000013
…,…,…,…,…,…
"""X""",154863124,"""C""","""T""","""X-154863124-C-T""",0.000011
"""X""",154863125,"""G""","""A""","""X-154863125-G-A""",0.000004
"""X""",154863151,"""C""","""T""","""X-154863151-C-T""",0.000004


### Merge the gnomad AF to VarChAMP alleles

In [67]:
gene_info_gnomad = gene_info.join(allele_freq_gnomad.lazy(),
                                  on="varid",
                                  how="left")

# gene_info_gnomad.to_csv("../3_outputs/varchamp_consolidated_alleles_gnomad_annot.tsv", sep="\t", index=False)

In [69]:
# gene_info_gnomad.collect()

### Merge the ClinVar info AND gnomad AF BOTH to VarChAMP alleles

In [79]:
gene_info_clinvar_gnomad = gene_info_clinvar.join(
    allele_freq_gnomad.select(pl.col(["varid","gnomad_af"])).lazy(),
    on="varid",how="left").collect()

gene_info_clinvar_gnomad.write_csv("../3_outputs/varchamp_consolidated_alleles_clinvar_gnomad.tsv", separator="\t")

## 4. Add Sequence Confirmation

In [99]:
## More stuff to be added

In [81]:
valid_sequence_confirmation_class = [1., 2., 7., 99.]

In [88]:
[col for col in gene_info.columns if col not in seq_confirm_res_df.columns]

<ipython-input-88-278c6c8e5a23>:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  [col for col in gene_info.columns if col not in seq_confirm_res_df.columns]


['chr',
 'chr_pos_38',
 'mutation_id_rc4',
 'mutation_id_cegs2',
 'mutation_id_edgotyping3',
 'entry_plate',
 'entry_well',
 'sequence',
 'chrom',
 'ref',
 'alt',
 'varid']

In [92]:
seq_confirm_res_df = pl.read_csv(f"../1_inputs/seq_confirmation_results/VarChampSeqConfirmationResult.tsv", separator="\t", infer_schema_length=10000)
seq_confirm_res_df = seq_confirm_res_df.drop_nulls(
    subset="entry_sequence_confirmation_class"
).rename(
    {"orf_id_wt": "orf_id",
     "mutation_id_old": "mut_id"}
)
seq_confirm_res_df

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,i64,str,i64,str,i64,str
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""","""2""","""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""","""2""","""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,113,"""CCSBVarC000003""","""ALE00000003""","""NC_000001.11:155237453:C:T""","""887G>A""","""Arg296Gln""","""RC4""","""RC4_Mut_GDEh1026""","""F01""","""GDEhDisVCh_40054""","""D12""","""2""","""RC4_Mut_GDDh1026""","""F01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""7""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,231,"""CCSBVarC000004""","""ALE00000004""","""NC_000001.11:155235252:A:G""","""1448T>C""","""Leu483Pro""","""RC4""","""RC4_Mut_GDEh1026""","""G01""","""GDEhDisVCh_40054""","""E12""","""2""","""RC4_Mut_GDDh1026""","""G01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,213510,"""CCSBVarC003869""","""ALE00003869""","""NC_000001.11:155239934:G:A""","""259C>T""","""Arg87Trp""","""CEGS2""","""CegsMutGDEh1035""","""B03""","""GDEhDisVCh_40054""","""B02""","""2""","""CegsMutGDDh1035""","""B03""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CTNNB1""","""ENSG00000168036""",100070227,212487,"""CCSBVarC007073""","""ALE00007071""","""NC_000003.12:41225785:A:G""","""860A>G""","""Asn287Ser""","""CEGS2""","""CegsMutGDEh1023""","""B07""","""GDEhDisVCh_40034""","""E09""","""2""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""TBX19""","""ENSG00000143178""",100070258,212488,"""CCSBVarC007074""","""ALE00007072""","""NC_000001.11:168291339:C:T""","""383C>T""","""Ser128Phe""","""CEGS2""","""CegsMutGDEh1043""","""A07""","""GDEhDisVCh_40046""","""B02""","""2""","""CegsMutGDDh1043""","""A07""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""7""",1,"""7""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""TBX19""","""ENSG00000143178""",100070258,212489,"""CCSBVarC007075""","""ALE00007073""","""NC_000001.11:168291213:T:G""","""257T>G""","""Met86Arg""","""CEGS2""","""CegsMutGDEh1043""","""G08""","""GDEhDisVCh_40046""","""F02""","""2""","""CegsMutGDDh1043""","""G08""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL"""


In [98]:
seq_confirm_res_df.filter(
    (pl.col("entry_sequence_confirmation_class")=="1")
    & (pl.col("entry_sequence_confirmation_class")!=pl.col("dualip_sequence_confirmation_class"))
    
)

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,i64,str,i64,str,i64,str
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""","""2""","""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""","""2""","""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,113,"""CCSBVarC000003""","""ALE00000003""","""NC_000001.11:155237453:C:T""","""887G>A""","""Arg296Gln""","""RC4""","""RC4_Mut_GDEh1026""","""F01""","""GDEhDisVCh_40054""","""D12""","""2""","""RC4_Mut_GDDh1026""","""F01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""7""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,231,"""CCSBVarC000004""","""ALE00000004""","""NC_000001.11:155235252:A:G""","""1448T>C""","""Leu483Pro""","""RC4""","""RC4_Mut_GDEh1026""","""G01""","""GDEhDisVCh_40054""","""E12""","""2""","""RC4_Mut_GDDh1026""","""G01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,213510,"""CCSBVarC003869""","""ALE00003869""","""NC_000001.11:155239934:G:A""","""259C>T""","""Arg87Trp""","""CEGS2""","""CegsMutGDEh1035""","""B03""","""GDEhDisVCh_40054""","""B02""","""2""","""CegsMutGDDh1035""","""B03""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRY""","""ENSG00000184895""",100070152,229149,"""CCSBVarC007069""","""ALE00007067""","""NC_000024.10:2787412:C:T""","""192G>A""","""Met64Ile""","""CEGS2""","""CegsMutGDEh1023""","""D05""","""GDEhDisVCh_40029""","""D09""","""1""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""SRY""","""ENSG00000184895""",100070152,229151,"""CCSBVarC007072""","""ALE00007070""","""NC_000024.10:2787267:C:T""","""337G>A""","""Ala113Thr""","""CEGS2""","""CegsMutGDEh1023""","""F09""","""GDEhDisVCh_40029""","""H09""","""1""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""SRY""","""ENSG00000184895""",100070152,229152,"""CCSBVarC007068""","""ALE00007066""","""NC_000024.10:2787207:G:A""","""397C>T""","""Arg133Trp""","""CEGS2""","""CegsMutGDEh1023""","""E09""","""GDEhDisVCh_40029""","""G09""","""1""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL""",0,"""NULL"""


In [82]:
gene_info_clinvar_gnomad = pl.read_csv("../3_outputs/varchamp_consolidated_alleles_clinvar_gnomad.tsv", separator="\t", infer_schema_length=10000)
seq_confirm_res_df = pl.read_csv(f"../1_inputs/seq_confirmation_results/VarChampSeqConfirmationResult.tsv", separator="\t", infer_schema_length=10000)
seq_confirm_res_df = seq_confirm_res_df.drop_nulls(
    subset="entry_sequence_confirmation_class"
).rename(
    {"orf_id_wt": "orf_id",
     "mutation_id_old": "mut_id"}
)
display(seq_confirm_res_df)
# print("seq_confirm_res_df unique ccsb_mutation_id", len(seq_confirm_res_df.dropna(subset="entry_sequence_confirmation_class")["ccsb_mutation_id"].unique()))

gene_info_clinvar_gnomad_sc = seq_confirm_res_df.join(
        gene_info_clinvar_gnomad, 
        on=["spdi"],
        suffix="_old"
)#.drop_nulls(subset="entry_sequence_confirmation_class")

gene_info_clinvar_gnomad_sc = gene_info_clinvar_gnomad_sc.drop(
    [col for col in gene_info_clinvar_gnomad_sc.columns if col.endswith("_old")]
).with_columns(
    pl.when(pl.col("entry_sequence_confirmation_class") == "NULL")
    .then(pl.lit(None))
    .otherwise(pl.col("entry_sequence_confirmation_class"))
    .alias("entry_sequence_confirmation_class")
).with_columns(
    pl.col("entry_sequence_confirmation_class").cast(pl.Float32).alias("entry_sequence_confirmation_class")
)

gene_info_clinvar_gnomad_sc = gene_info_clinvar_gnomad_sc.filter(pl.col("entry_sequence_confirmation_class").is_in(valid_sequence_confirmation_class))
gene_info_clinvar_gnomad_sc.write_csv("../3_outputs/varchamp_consolidated_alleles_clinvar_gnomad_sequence_confirmed.tsv", separator="\t")

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,i64,str,i64,str,i64,str
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""","""2""","""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""","""2""","""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,113,"""CCSBVarC000003""","""ALE00000003""","""NC_000001.11:155237453:C:T""","""887G>A""","""Arg296Gln""","""RC4""","""RC4_Mut_GDEh1026""","""F01""","""GDEhDisVCh_40054""","""D12""","""2""","""RC4_Mut_GDDh1026""","""F01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""7""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,231,"""CCSBVarC000004""","""ALE00000004""","""NC_000001.11:155235252:A:G""","""1448T>C""","""Leu483Pro""","""RC4""","""RC4_Mut_GDEh1026""","""G01""","""GDEhDisVCh_40054""","""E12""","""2""","""RC4_Mut_GDDh1026""","""G01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""GBA1""","""ENSG00000177628""",2,213510,"""CCSBVarC003869""","""ALE00003869""","""NC_000001.11:155239934:G:A""","""259C>T""","""Arg87Trp""","""CEGS2""","""CegsMutGDEh1035""","""B03""","""GDEhDisVCh_40054""","""B02""","""2""","""CegsMutGDDh1035""","""B03""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CTNNB1""","""ENSG00000168036""",100070227,212487,"""CCSBVarC007073""","""ALE00007071""","""NC_000003.12:41225785:A:G""","""860A>G""","""Asn287Ser""","""CEGS2""","""CegsMutGDEh1023""","""B07""","""GDEhDisVCh_40034""","""E09""","""2""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""TBX19""","""ENSG00000143178""",100070258,212488,"""CCSBVarC007074""","""ALE00007072""","""NC_000001.11:168291339:C:T""","""383C>T""","""Ser128Phe""","""CEGS2""","""CegsMutGDEh1043""","""A07""","""GDEhDisVCh_40046""","""B02""","""2""","""CegsMutGDDh1043""","""A07""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""7""",1,"""7""",0,"""NULL""",0,"""NULL""",0,"""NULL"""
"""TBX19""","""ENSG00000143178""",100070258,212489,"""CCSBVarC007075""","""ALE00007073""","""NC_000001.11:168291213:T:G""","""257T>G""","""Met86Arg""","""CEGS2""","""CegsMutGDEh1043""","""G08""","""GDEhDisVCh_40046""","""F02""","""2""","""CegsMutGDDh1043""","""G08""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,"""1""",1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL"""


In [83]:
gene_info_clinvar_gnomad_sc

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class,chr,chr_pos_38,mutation_id_rc4,mutation_id_cegs2,…,nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean,gnomad_af
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,f32,i64,str,i64,str,i64,str,i64,str,str,i64,str,str,…,str,str,str,str,str,str,str,str,str,i64,i64,str,str,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,f64
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""","""2""","""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1.0,1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL""","""1""",155240033,"""6""","""NULL""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""","""2""","""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1.0,1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL""","""1""",155238225,"""73""","""NULL""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,113,"""CCSBVarC000003""","""ALE00000003""","""NC_000001.11:155237453:C:T""","""887G>A""","""Arg296Gln""","""RC4""","""RC4_Mut_GDEh1026""","""F01""","""GDEhDisVCh_40054""","""D12""","""2""","""RC4_Mut_GDDh1026""","""F01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1.0,1,"""7""",0,"""NULL""",0,"""NULL""",0,"""NULL""","""1""",155237453,"""113""","""NULL""",…,"""-""","""RCV000004573|RCV000079357|RCV0…","""MONDO:MONDO:0011945,MedGen:C18…","""Gaucher disease perinatal leth…","""germline;unknown""","""germline""","""GRCh38""","""NC_000001.11""","""1""",155237453,155237453,"""1q22""","""criteria provided, multiple su…",15,"""-""","""N""","""ClinGen:CA221417,UniProtKB:P04…",3,4328,155237453,"""C""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000232587|SCV000321701|SCV0…","""-""","""-""","""887G>A ""","""Arg296Gln""","""NM_000157.4""",2,"""1_Pathogenic""",null
"""GBA1""","""ENSG00000177628""",2,231,"""CCSBVarC000004""","""ALE00000004""","""NC_000001.11:155235252:A:G""","""1448T>C""","""Leu483Pro""","""RC4""","""RC4_Mut_GDEh1026""","""G01""","""GDEhDisVCh_40054""","""E12""","""2""","""RC4_Mut_GDDh1026""","""G01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","